In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '')

# PatchCore — features pré-entraînées (alternative à l'auto-encodeur)

Suite de [tp_deep_learning.ipynb](tp_deep_learning.ipynb) (voir aussi le
[rapport](tp_deep_learning_report.md)) : au lieu d'apprendre à **reconstruire** les images, on compare les
features d'un CNN **pré-entraîné** (gelé, non ré-entraîné sur nos données) à une mémoire de patchs saines.
Le score d'anomalie d'un patch = distance à son plus proche voisin dans cette mémoire ; le score d'une
image = le pire de ses patchs.

## Chargement des données

**Pas** les `.bin.gz` de la partie 2 (ceux-ci passent par le pipeline 128×128 de l'auto-encodeur — un
double redimensionnement 1024→128→224 qui détruirait du détail fin dès la première étape, alors que
PatchCore n'a aucune contrainte de taille fixe). On recharge directement les PNG originaux et on
redimensionne **une seule fois**, 1024 → 224.

In [2]:
from pathlib import Path

import numpy as np

from src.vision.data import build_defect_index, list_defect_types, load_folder, load_mask, load_test_defects_by_category

DATA_DIR = Path('deep-learning/screw')
BACKBONE_SIZE = 224
SEED = 42

TRAIN_GOOD_DIR = DATA_DIR / 'train' / 'good'
TEST_DIR = DATA_DIR / 'test'
GT_DIR = DATA_DIR / 'ground_truth'
defect_types = list_defect_types(TEST_DIR)

X_train_norm, _ = load_folder(TRAIN_GOOD_DIR, BACKBONE_SIZE, normalize=True)
test_good_norm, _ = load_folder(TEST_DIR / 'good', BACKBONE_SIZE, normalize=True)

# Concatène les défauts dans l'ordre des catégories (triées) puis des fichiers (triés) — et garde la
# catégorie ainsi que le chemin du masque ground-truth de chaque indice au passage.
defects_by_category = load_test_defects_by_category(TEST_DIR, defect_types, BACKBONE_SIZE, normalize=True)
test_defects_norm_stacked = np.concatenate([defects_by_category[d][0] for d in defect_types])
defect_category_per_index, mask_paths = build_defect_index(TEST_DIR, GT_DIR, defect_types)

print('Train (normé)  :', X_train_norm.shape, X_train_norm.dtype)
print('Test saines    :', test_good_norm.shape)
print('Test défauts   :', test_defects_norm_stacked.shape)

Train (normé)  : (320, 224, 224, 1) float32
Test saines    : (41, 224, 224, 1)
Test défauts   : (119, 224, 224, 1)


## Extraction de features

`ResNet50` pré-entraîné sur ImageNet, tronqué à une couche intermédiaire (`conv3_block4_out`, stride 8 →
grille 28×28 — retour à la résolution fine maintenant que la mémoire est complète, pour voir si elle aide
une fois la couverture mémoire contrôlée). Nos images sont en niveaux de gris (224×224×1) : on les
convertit en RGB avant de les passer dans le backbone (gelé — pas d'entraînement).

In [3]:
from src.vision.patchcore import build_feature_extractor, extract_patch_features

LAYER_NAME = 'conv3_block4_out'  # stride 8 -> grille 28x28 pour une entrée 224x224

feature_extractor = build_feature_extractor(BACKBONE_SIZE, LAYER_NAME)

sample = extract_patch_features(X_train_norm[:1], feature_extractor, BACKBONE_SIZE)
print('Grille de patchs :', sample.shape[1:3], '| dimension des features :', sample.shape[-1])

Grille de patchs : (28, 28) | dimension des features : 512


## Mémoire de patchs saines

On extrait les features de **toutes** les images saines d'entraînement, on aplatit en patchs. `MEMORY_SIZE`
plafonne un éventuel sous-échantillonnage — ici fixé au-delà du nombre total de patchs (250 880 avec la
grille 28×28), donc **mémoire complète, sans sous-échantillonnage**.

*Constat empirique* : avec seulement 4-16% des patchs en mémoire, l'AUROC plafonnait à ~0.75 quelle que
soit la résolution de la grille (28×28 ou 14×14) — beaucoup de patchs saines de test n'avaient simplement
aucun voisin proche dans une mémoire trop clairsemée. Avec la mémoire complète, la résolution redevient
discriminante : **AUROC 0.923 en 14×14, 0.973 en 28×28** (grille fine). Les deux facteurs comptent donc,
mais la couverture mémoire est la condition *préalable* — sans elle, une meilleure résolution ne se voit
pas. Résultat final (28×28, mémoire complète) : quasi à égalité avec l'auto-encodeur (AUROC 0.982), et même
moins de faux négatifs (8 contre 16 sur 119 défauts).

In [4]:
from src.vision.patchcore import build_memory_bank

MEMORY_SIZE = 300_000  # au-delà du total de patchs (28x28 grid) -> mémoire complète, pas de sous-échantillonnage

train_features = extract_patch_features(X_train_norm, feature_extractor, BACKBONE_SIZE)
memory_bank, total_patches = build_memory_bank(train_features, MEMORY_SIZE, SEED)

print(f'Mémoire : {memory_bank.shape} (sous-échantillonnée depuis {total_patches} patchs)')

Mémoire : (250880, 512) (sous-échantillonnée depuis 250880 patchs)


## Score d'anomalie (plus proche voisin)

Pour chaque patch d'une image de test, on cherche son plus proche voisin dans la mémoire de patchs saines.
Le score de l'image = la **pire** distance parmi ses patchs (un seul patch anormal suffit à signaler
l'image, contrairement à une moyenne qui dilue le signal — même logique que le centile 99 côté
auto-encodeur).

In [5]:
from sklearn.neighbors import NearestNeighbors

from src.vision.patchcore import anomaly_score

nearest_neighbor = NearestNeighbors(n_neighbors=1).fit(memory_bank)

In [6]:
errors_test_good, patch_maps_test_good = anomaly_score(test_good_norm, feature_extractor, nearest_neighbor, BACKBONE_SIZE)
errors_test_defects, patch_maps_test_defects = anomaly_score(test_defects_norm_stacked, feature_extractor, nearest_neighbor, BACKBONE_SIZE)

print('Score (saines, test)  :', errors_test_good.shape)
print(f'  moyenne = {errors_test_good.mean():.3f}, écart-type = {errors_test_good.std():.3f}')
print('Score (défauts, test) :', errors_test_defects.shape)
print(f'  moyenne = {errors_test_defects.mean():.3f}, écart-type = {errors_test_defects.std():.3f}')

Score (saines, test)  : (41,)
  moyenne = 16.937, écart-type = 1.358
Score (défauts, test) : (119,)
  moyenne = 22.169, écart-type = 2.559


## Distribution des scores

In [7]:
from src.vision.visualization import plot_score_histogram

threshold_p95 = np.percentile(errors_test_good, 95)

plot_score_histogram(
    errors_test_good, threshold_p95,
    title='Distribution du score (saines, test)',
    xlabel="score d'anomalie (distance au plus proche voisin)",
    threshold_label='seuil (centile 95, test saines)',
)

/Users/kenji/formation-ia/indusense_tp/src/vision/visualization.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
plot_score_histogram(
    errors_test_defects, threshold_p95, color='tab:orange',
    title='Distribution du score (défauts, test)',
    xlabel="score d'anomalie (distance au plus proche voisin)",
    threshold_label='seuil (centile 95, test saines)',
)

## ROC & matrice de confusion

In [9]:
from src.vision.evaluation import plot_roc_and_confusion

y_true = np.concatenate([np.zeros(len(errors_test_good)), np.ones(len(errors_test_defects))])
y_score = np.concatenate([errors_test_good, errors_test_defects])

roc_auc, y_pred = plot_roc_and_confusion(
    y_true, y_score, threshold_p95, threshold_label='seuil centile 95, test saines',
)

/Users/kenji/formation-ia/indusense_tp/src/vision/evaluation.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/kenji/formation-ia/indusense_tp/src/vision/evaluation.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Quel défaut est loupé ?

In [10]:
from src.vision.evaluation import missed_defect_counts, plot_missed_defects

n_good = len(errors_test_good)
counts, fn_idx = missed_defect_counts(y_true, y_pred, defect_category_per_index, n_good)
plot_missed_defects(counts, defect_types)

manipulated_front   : 0 loupé(s)
scratch_head        : 0 loupé(s)
scratch_neck        : 0 loupé(s)
thread_side         : 6 loupé(s)
thread_top          : 2 loupé(s)


/Users/kenji/formation-ia/indusense_tp/src/vision/evaluation.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Visualisation des faux négatifs restants

`original | heatmap (carte de distance par patch) | masque ground-truth` — pas de "reconstruction" ici
(PatchCore n'en produit pas), la heatmap est directement la carte de distances au plus proche voisin,
patch par patch (28×28, avant tout redimensionnement).

In [11]:
from src.vision.visualization import plot_image_grid

rows, row_labels = [], []
for idx in fn_idx:
    original = test_defects_norm_stacked[idx, ..., 0]
    heatmap = patch_maps_test_defects[idx]
    mask = load_mask(mask_paths[idx], BACKBONE_SIZE)
    rows.append([original, heatmap, mask])
    row_labels.append(defect_category_per_index[idx])

col_titles = ['original', 'heatmap (distance patch)', 'masque ground-truth']
plot_image_grid(
    rows, 'Faux négatifs (défauts prédits saine) — PatchCore',
    col_titles=col_titles, row_labels=row_labels, cmaps=['gray', 'inferno', 'gray'],
)

/Users/kenji/formation-ia/indusense_tp/src/vision/visualization.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
